# 01 · Prepare data  *(branch `exp/02-leakage-fixes`)*

Turns the 1-minute dataset into everything the later notebooks use. Runs once (~5 min, CPU is fine).

| Output (on Drive, `thesis_rl/prepared/`) | |
|---|---|
| `alpha101_{train,val,test}.npy` | 101 Alpha101 factors, normalised with **train-fitted** 1 %/99 % quantiles |
| `labels_{train,val,test}.npy` | RNN targets, one set **per split** (thresholds fitted on train) |
| `market_{train,val,test}.npz` | prices + LLM signals (**lagged one day**; same-day copies kept for the leak study only) |
| `meta.json` | split boundaries and dates |

Leakage fixes applied here (details in `LEAKAGE_AUDIT.md`): per-split labels, no `bfill` in `decay_linear`,
factors computed once on the continuous series with train-only normalisation, one-day signal lag.

**Before running:** upload `BTC_1min_with_sentiment_risk_train.csv.gz` to `MyDrive/thesis_rl/data/`.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/02-leakage-fixes"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
seeding.record_environment(L.report / "environment", "01_prepare_data")   # exact environment of this run (pip freeze, CUDA, GPU)

## 1 · Checks before touching the data
The three leakage tests. The Alpha101 test compares all 101 factors against the notebook's original helpers on 1,200 real bars.

In [ ]:
assert L.data.exists(), f"upload the dataset to {L.data}"
env = {**os.environ, "BTC_CSV": str(L.data)}
for t in ("test_causality.py", "test_signal_lag.py", "test_alpha101_equivalence.py"):
    print(f"--- {t}")
    r = subprocess.run([sys.executable, f"tests/{t}"], env=env, capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:] if r.returncode else "")
    assert r.returncode == 0, f"{t} failed - do not continue"

## 2 · Prepare

In [ ]:
from rltrade.prepare import prepare
FORCE = False                     # True re-builds everything
if FORCE or not (L.prepared / "meta.json").exists():
    meta = prepare(L.data, L.prepared)
else:
    print("already prepared - set FORCE = True to rebuild")

## 3 · Sanity checks

In [ ]:
import json, numpy as np, pandas as pd
meta = json.loads((L.prepared / "meta.json").read_text())
print(pd.DataFrame(meta["splits"]).T)
for s in ("train", "val", "test"):
    a = np.load(L.prepared / f"alpha101_{s}.npy"); y = np.load(L.prepared / f"labels_{s}.npy")
    m = np.load(L.prepared / f"market_{s}.npz")
    print(f"{s:5} alpha {a.shape} in [{a.min():.2f},{a.max():.2f}] | labels {y.shape} mean {y.mean():+.3f} "
          f"| sentiment lagged {m['sentiment_score'][:3]} vs same-day {m['sameday__sentiment_score'][:3]}")